In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from itertools import combinations
from collections import defaultdict

In [2]:
from src.utils import path


In [3]:
def build_vertical_db(transactions):
    """Construye la base de datos vertical (TID-list) a partir de las transacciones."""
    vertical_db = defaultdict(set)
    for tid, transaction in enumerate(transactions):
        for item in transaction:
            vertical_db[frozenset([item])].add(tid)
    return dict(vertical_db)


def eclat(prefix, items, min_support, frequent_itemsets, n_transactions):
    """Búsqueda recursiva en profundidad de itemsets frecuentes."""
    while items:
        item, tid_set = items.pop(0)
        new_prefix = prefix | item
        support = len(tid_set)

        if support >= min_support:
            frequent_itemsets[new_prefix] = {
                "support_count": support,
                "support_pct": round(support / n_transactions, 4),
            }
            suffix = []
            for other_item, other_tid_set in items:
                new_tid_set = tid_set & other_tid_set
                if len(new_tid_set) >= min_support:
                    suffix.append((other_item, new_tid_set))
            if suffix:
                eclat(new_prefix, suffix, min_support,
                      frequent_itemsets, n_transactions)


def run_eclat(transactions, min_support=0.3):
    """Ejecuta ECLAT y devuelve los itemsets frecuentes."""
    n = len(transactions)
    min_sup_count = max(1, int(min_support * n))
    vertical_db = build_vertical_db(transactions)

    frequent_1 = {
        item: tids
        for item, tids in vertical_db.items()
        if len(tids) >= min_sup_count
    }

    frequent_itemsets = {
        item: {"support_count": len(tids),
               "support_pct": round(len(tids) / n, 4)}
        for item, tids in frequent_1.items()
    }

    items_list = sorted(frequent_1.items(), key=lambda x: str(sorted(x[0])))
    eclat(frozenset(), items_list, min_sup_count, frequent_itemsets, n)
    return frequent_itemsets


def generate_rules(frequent_itemsets, min_confidence=0.6):
    """Genera reglas de asociación a partir de los itemsets frecuentes."""
    rules = []
    for itemset, stats in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        for size in range(1, len(itemset)):
            for antecedent in combinations(sorted(itemset), size):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent
                if antecedent in frequent_itemsets:
                    ant_sup = frequent_itemsets[antecedent]["support_count"]
                    confidence = stats["support_count"] / ant_sup
                    if confidence >= min_confidence:
                        con_sup_pct = frequent_itemsets.get(
                            consequent, {}).get("support_pct")
                        lift = round(confidence / con_sup_pct, 4) \
                            if con_sup_pct else None
                        rules.append({
                            "antecedents": set(antecedent),
                            "consequents": set(consequent),
                            "support":     stats["support_pct"],
                            "confidence":  round(confidence, 4),
                            "lift":        lift,
                        })
    return sorted(rules, key=lambda r: r["confidence"], reverse=True)


def results_to_df(frequent_itemsets, rules):
    """Convierte los resultados a DataFrames de pandas."""
    df_items = pd.DataFrame([
        {
            "itemset": ", ".join(sorted(k)),
            "tamaño":  len(k),
            "support_count": v["support_count"],
            "support": v["support_pct"],
        }
        for k, v in frequent_itemsets.items()
    ]).sort_values(["tamaño", "support_count"], ascending=[True, False])

    df_rules = pd.DataFrame([
        {
            "antecedents": frozenset(r["antecedents"]),
            "consequents": frozenset(r["consequents"]),
            "support":     r["support"],
            "confidence":  r["confidence"],
            "lift":        r["lift"],
        }
        for r in rules
    ])

    return df_items, df_rules

In [4]:
directorio_proyecto = path.obtener_ruta_local()
datos = pd.read_csv(directorio_proyecto+'\\data\\procesed\\datos_tourism.csv',delimiter = ',',decimal = ".", encoding='utf-8')
datos.head()

,Interests,Accessibility,Site Name,Sites Visited,Age_Group,Tour_Duration_Pref,Rating_Category,Response_Speed,Accuracy_Level,VR_Quality,Satisfaction_Level
0,"Architecture, Art, History",False,Eiffel Tower,"Eiffel Tower, Great Wall of China, Taj Mahal",46-55,Media(5-7h),Bajo,Lento,Alta,Buena,Media
1,"Cultural, Nature",False,Colosseum,Great Wall of China,36-45,Media(5-7h),Medio,Normal,Media,Buena,Media
2,"History, Art, Architecture",True,Machu Picchu,Eiffel Tower,36-45,Media(5-7h),Bajo,Rápido,Media,Excelente,Media
3,"Cultural, Art, Architecture",False,Colosseum,"Machu Picchu, Taj Mahal",46-55,Larga(8-10h),Bajo,Rápido,Media,Excelente,Media
4,"Architecture, Art",True,Colosseum,"Machu Picchu, Taj Mahal, Great Wall of China",46-55,Media(5-7h),Alto,Rápido,Alta,Excelente,Alta


In [5]:
df_paquete_ventas = datos[["Sites Visited"]]
df_paquete_ventas.head()

,Sites Visited
0,"Eiffel Tower, Great Wall of China, Taj Mahal"
1,Great Wall of China
2,Eiffel Tower
3,"Machu Picchu, Taj Mahal"
4,"Machu Picchu, Taj Mahal, Great Wall of China"


In [6]:
df_paquete_ventas.shape


(5000, 1)

In [7]:
data_paquete = list(
    df_paquete_ventas["Sites Visited"].apply(lambda x: [i.strip() for i in str(x).split(",")])
)
data_paquete[:3]

[['Eiffel Tower', 'Great Wall of China', 'Taj Mahal'],
 ['Great Wall of China'],
 ['Eiffel Tower']]

In [8]:
# Establezca un valor umbral para el soporte mínimo y ejecute ECLAT
frequent_itemsets_paquete = run_eclat(data_paquete, min_support=0.2)
df_itemsets_paquete, _ = results_to_df(frequent_itemsets_paquete, [])
df_itemsets_paquete

,itemset,tamaño,support_count,support
0,Eiffel Tower,1,2072,0.4144
2,Taj Mahal,1,2012,0.4024
4,Colosseum,1,1994,0.3988
1,Great Wall of China,1,1991,0.3982
3,Machu Picchu,1,1920,0.3840


In [9]:
# Reglas de asociación con umbral de confianza mínima
rules_paquete = generate_rules(frequent_itemsets_paquete, min_confidence=0.2)
_, df_ar_paquete = results_to_df(frequent_itemsets_paquete, rules_paquete)
df_ar_paquete

""


In [11]:
# Scatter plot de confidence vs lift
plt.figure(figsize=(8, 6))
plt.scatter(df_ar_paquete["confidence"], df_ar_paquete["lift"],
            alpha=0.6, c="steelblue", edgecolors="k", s=60)
plt.xlabel("Confidence")
plt.ylabel("Lift")
plt.title("Reglas de Asociación ECLAT – Visited Sites")
plt.tight_layout()
plt.show()

KeyError: 'confidence'

<Figure size 800x600 with 0 Axes>